In [2]:
# ============================================================
# Experiment 4 — Comparative Study of CNN Architectures
# LeNet-5, AlexNet, VGG16, GoogleNet (InceptionV3 stand-in), ResNet50
#
# - VGG16 / ResNet50 / InceptionV3: real transfer learning
#   (ImageNet weights, frozen conv base, same head as Task 2/3)
# - LeNet-5 / AlexNet: no pretrained weights exist anywhere for these,
#   so they are built + trained from scratch, per the manual's
#   instruction to "handle appropriately" when TL isn't available.
# - GoogleNet itself is not in keras.applications; InceptionV3 is used
#   as the closest available stand-in — flag this in your report.
#
# Uses the SAME hyperparameters as your Task 3 (Adam, lr=0.001,
# batch_size=32) for a fair, single-configuration comparison across
# architectures — this is NOT a hyperparameter sweep.
# ============================================================

import gc
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import (
    Input, Conv2D, AveragePooling2D, MaxPooling2D, Flatten, Dense,
    Dropout, GlobalAveragePooling2D, Resizing
)
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import VGG16, ResNet50, InceptionV3
from tensorflow.keras import backend as K

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# ------------------------------------------------------------
# 1. Load & preprocess CIFAR-10 (same as your Task 1)
# ------------------------------------------------------------
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer',
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

print("Training data shape:", x_train.shape)
print("Testing data shape:", x_test.shape)



170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1992s 12us/step
Training data shape: (50000, 32, 32, 3)
Testing data shape: (10000, 32, 32, 3)


In [3]:
# ------------------------------------------------------------
# Common hyperparameters — same as manual's Task 3
# ------------------------------------------------------------
LR = 0.001
BATCH_SIZE = 32
EPOCHS = 10   # drop to 5 if you just need the comparison table faster

# ------------------------------------------------------------
# 2. Model builders
# ------------------------------------------------------------

def build_lenet5():
    """No pretrained ImageNet weights exist for LeNet-5 (predates ImageNet).
    Trained from scratch, adapted from grayscale 28x28 to CIFAR-10's 32x32x3."""
    model = Sequential([
        Input(shape=(32, 32, 3)),
        Conv2D(6, (5, 5), activation='relu', padding='same'),
        AveragePooling2D((2, 2)),
        Conv2D(16, (5, 5), activation='relu'),
        AveragePooling2D((2, 2)),
        Flatten(),
        Dense(120, activation='relu'),
        Dense(84, activation='relu'),
        Dense(10, activation='softmax')
    ], name="LeNet5")
    return model


def build_alexnet():
    """No pretrained weights exist for AlexNet in keras.applications either.
    Trained from scratch; scaled down from the original 227x227 input to
    fit CIFAR-10's 32x32 while keeping the same conv-stage progression."""
    model = Sequential([
        Input(shape=(32, 32, 3)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Conv2D(192, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Conv2D(384, (3, 3), activation='relu', padding='same'),
        Conv2D(256, (3, 3), activation='relu', padding='same'),
        Conv2D(256, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(1024, activation='relu'),
        Dropout(0.5),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(10, activation='softmax')
    ], name="AlexNet")
    return model


def build_transfer_model(base_class, target_size, name):
    """Generic transfer-learning builder for architectures that DO have
    ImageNet pretrained weights: VGG16, ResNet50, InceptionV3 (GoogleNet
    stand-in). Resizes CIFAR-10's 32x32 up to each backbone's minimum
    accepted input size, then freezes the conv base — same recipe as
    your Task 2/3."""
    inputs = Input(shape=(32, 32, 3))
    x = Resizing(target_size, target_size)(inputs)
    base_model = base_class(weights='imagenet', include_top=False, input_tensor=x)
    base_model.trainable = False  # freeze convolutional base

    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(256, activation='relu')(x)
    outputs = Dense(10, activation='softmax')(x)

    return Model(inputs=inputs, outputs=outputs, name=name)

# ------------------------------------------------------------
# 3. Train + evaluate helper
# ------------------------------------------------------------

def train_and_evaluate(model, name):
    model.compile(optimizer=Adam(learning_rate=LR),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    start = time.time()
    history = model.fit(
        x_train, y_train_cat,
        validation_data=(x_test, y_test_cat),
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1
    )
    train_time = time.time() - start

    y_pred_probs = model.predict(x_test, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = y_test.flatten()

    test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    result = {
        'Model': name,
        'Parameters': model.count_params(),
        'Train Accuracy': round(history.history['accuracy'][-1], 4),
        'Test Accuracy (%)': round(test_acc * 100, 2),
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1-score': round(f1, 4),
        'Training Time (s)': round(train_time, 2)
    }

    return result, history, cm

# ------------------------------------------------------------
# 4. Run all 5 architectures, one config each
# ------------------------------------------------------------
all_results = []
histories = {}
conf_matrices = {}

model_configs = [
    ("LeNet-5",                build_lenet5,       None),
    ("AlexNet",                build_alexnet,      None),
    ("VGG16",                  lambda: build_transfer_model(VGG16, 32, "VGG16"), None),
    ("GoogleNet (InceptionV3)",lambda: build_transfer_model(InceptionV3, 75, "GoogleNet"), None),
    ("ResNet50",                lambda: build_transfer_model(ResNet50, 32, "ResNet50"), None),
]

for name, builder, _ in model_configs:
    print(f"\n{'='*60}\nTraining: {name}\n{'='*60}")
    model = builder()
    result, history, cm = train_and_evaluate(model, name)
    all_results.append(result)
    histories[name] = history
    conf_matrices[name] = cm

    # free memory before building the next (large) model
    del model
    K.clear_session()
    gc.collect()

# ------------------------------------------------------------
# 5. Final comparison table — matches manual Section 18.2
# ------------------------------------------------------------
comparison_df = pd.DataFrame(all_results)
comparison_df['Parameters (M)'] = (comparison_df['Parameters'] / 1e6).round(2)
comparison_df = comparison_df[[
    'Model', 'Parameters (M)', 'Train Accuracy', 'Test Accuracy (%)',
    'Precision', 'Recall', 'F1-score', 'Training Time (s)'
]]

print("\n\nFinal Comparison Table:")
print(comparison_df.to_string(index=False))

comparison_df
# `histories` and `conf_matrices` are kept per-model if you want to
# generate the mandatory accuracy/loss/confusion-matrix plots per
# architecture (reuse your existing plotting cells from Task 5,
# looping over histories.items() / conf_matrices.items()).


Training: LeNet-5
Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - accuracy: 0.4100 - loss: 1.6297 - val_accuracy: 0.4866 - val_loss: 1.4421
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.5156 - loss: 1.3621 - val_accuracy: 0.5330 - val_loss: 1.3146
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5587 - loss: 1.2413 - val_accuracy: 0.5399 - val_loss: 1.2985
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.5927 - loss: 1.1515 - val_accuracy: 0.5766 - val_loss: 1.1904
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.6187 - loss: 1.0822 - val_accuracy: 0.5818 - val_loss: 1.1845
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.6384 - loss: 1.0203 - val_accuracy: 0.6011 - val_loss: 1.1416
Epoch 7/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.6591 - loss: 0.9638 - val_accuracy: 0.6074 - val_loss: 1.1177
Epoch 8/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.67

,Model,Parameters (M),Train Accuracy,Test Accuracy (%),Precision,Recall,F1-score,Training Time (s)
0,LeNet-5,0.08,0.7068,61.03,0.6204,0.6103,0.6132,66.11
1,AlexNet,6.98,0.7981,72.09,0.7214,0.7209,0.7180,201.27
2,VGG16,14.85,0.7027,61.01,0.6200,0.6101,0.6098,152.42
3,GoogleNet (InceptionV3),22.33,0.9509,60.32,0.6022,0.6032,0.6010,268.28
4,ResNet50,24.11,0.3915,40.80,0.4163,0.4080,0.3948,172.55
